In [ ]:
!pip install --upgrade MNN

import numpy as np
import MNN
import cv2

In [85]:
def load_labels(filename):
	with open(filename, 'r') as f:
		return [line.strip() for line in f.readlines()]

class Classifier():
  def __init__(self, model, labels):
    self.interpreter = MNN.Interpreter(model)
    self.session = self.interpreter.createSession()
    self.input_tensor = self.interpreter.getSessionInput(self.session)
    self.labels = load_labels(labels)

  def predict(self, image):
    # change to rgb format
    image = image[..., ::-1]
    image = cv2.resize(image, (224, 224))
    image = image.astype(np.float32)

    # preprocess image
    image = image - np.array([127.5,127.5,127.5], dtype=np.float32)
    image = image * np.array([0.00784,0.00784,0.00784], dtype=np.float32)

    # NHWC → NCHW
    image = image.transpose((2, 0, 1))
    image = np.expand_dims(image, 0)
    image = np.ascontiguousarray(image, dtype=np.float32)

    tmp_input = MNN.Tensor(
      (1,3,224,224),
      MNN.Halide_Type_Float,
      image,
      MNN.Tensor_DimensionType_Caffe
    )

    self.input_tensor.copyFrom(tmp_input)
    self.interpreter.runSession(self.session)

    output_tensor = self.interpreter.getSessionOutput(self.session)
    preds = output_tensor.getData()

    preds = np.array(preds)

    ix = preds.argmax()

    return (self.labels[ix], str(float(preds[ix])))


In [86]:
objects = []

net = cv2.dnn_DetectionModel('/content/yolov4/yolov4.cfg', '/content/yolov4/yolov4.weights')
net.setInputSize(608, 608)
net.setInputScale(1.0 / 255)
net.setInputSwapRB(True)

car_make_classifier = Classifier('/content/model-weights-spectrico-car-brand-recognition-mobilenet_v3-224x224-170620.mnn', '/content/labels-makes.txt')
car_color_classifier = Classifier('/content/model-weights-spectrico-car-colors-recognition-mobilenet_v3-224x224-180420.mnn', '/content/labels-colors.txt')

img = cv2.imread("/content/EBZ3A001772895187790.jpg")

h, w, _ = img.shape

classes, confidences, boxes = net.detect(img, confThreshold=0.1, nmsThreshold=0.4)

In [87]:
for classId, confidence, box in zip(classes.flatten(), confidences.flatten(), boxes):
  if classId in [2,5,7] and confidence > 0.3:
    left, top, width, height = box

    x1 = max(0, left)
    y1 = max(0, top)
    x2 = min(w, left + width)
    y2 = min(h, top + height)

    car_img = img[y1:y2, x1:x2]

    if car_img.size == 0:
        continue

    (make, make_confidence) = car_make_classifier.predict(car_img)
    (color, color_confidence) = car_color_classifier.predict(car_img)

    print(make)
    print(color)

    rect = {
      "left": str(x1),
      "top": str(y1),
      "width": str(x2-x1),
      "height": str(y2-y1)
    }

    objects.append({
      "make": make,
      "color": color,
      "make_prob": str(make_confidence),
      "color_prob": str(color_confidence),
      "obj_prob": str(confidence),
      "rect": rect
    })

print(objects)

Jaguar
black
[{'make': 'Jaguar', 'color': 'black', 'make_prob': '0.9681873917579651', 'color_prob': '0.9955993890762329', 'obj_prob': '0.9934258', 'rect': {'left': '1', 'top': '27', 'width': '979', 'height': '589'}}]
